# 03 — PU-Bagging Model

**Задача:** Positive-Unlabeled Learning — найти скрытых предпринимателей среди consumer-карт.

**Пайплайн:**
1. Baseline: Logistic Regression
2. Основная модель: PU-Bagging + LightGBM (N=10 итераций)
3. Challenger: PU-Bagging + CatBoost
4. Валидация: PR-AUC, ROC-AUC, Confusion Matrix, Precision@K
5. Synthetic Injection Test (оценка recall)
6. SHAP: объяснение модели

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.models.train import load_feature_matrix, prepare_splits, save_model, load_model
from src.models.baseline import build_baseline
from src.models.pu_bagging import PUBaggingClassifier
from src.models.catboost_model import PUBaggingCatBoost
from src.evaluation.metrics import evaluate_on_holdout, precision_at_k_table
from src.evaluation.plots import (
    plot_pr_curve, plot_roc_curve, plot_confusion_matrix,
    plot_score_distribution, plot_precision_at_k, plot_feature_importance,
)
from src.evaluation.injection_test import run_injection_test
from src.config import DEFAULT_THRESHOLD, MODELS_DIR

MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Загрузка данных и сплит

In [ ]:
fm = load_feature_matrix()
X_pos_train, X_pos_holdout, X_unlabeled, y_holdout, feature_cols = prepare_splits(fm)

print(f"Positives (train):   {len(X_pos_train):,}")
print(f"Positives (holdout): {len(X_pos_holdout):,}")
print(f"Unlabeled consumer:  {len(X_unlabeled):,}")
print(f"Features:            {len(feature_cols)}")
print(f"\nFeatures used: {feature_cols[:10]} ...")

# y_holdout_combined: holdout biz=1 + consumer=0 (для метрик)
y_holdout_all = np.concatenate([np.ones(len(X_pos_holdout)), np.zeros(len(X_unlabeled))])
X_holdout_all = pd.concat([X_pos_holdout, X_unlabeled], ignore_index=True)

## 2. Baseline: Logistic Regression

In [ ]:
y_baseline_train = np.concatenate([
    np.ones(len(X_pos_train)),
    np.zeros(len(X_unlabeled)),
])
X_baseline_train = pd.concat([X_pos_train, X_unlabeled], ignore_index=True)

baseline = build_baseline(X_baseline_train, y_baseline_train)
save_model(baseline, "baseline_logreg")

scores_baseline = baseline.predict_proba(X_holdout_all)[:, 1]
metrics_baseline = evaluate_on_holdout(y_holdout_all, scores_baseline, label="LogReg Baseline")

## 3. PU-Bagging + LightGBM (основная модель)

~10-15 минут на 10 итерациях.

In [ ]:
pu_lgbm = PUBaggingClassifier(n_iterations=10, verbose=True)
pu_lgbm.fit(X_pos_train, X_unlabeled)
save_model(pu_lgbm, "pu_bagging_lgbm")

# Скоры на holdout (biz + consumer)
scores_lgbm = pu_lgbm.predict_proba_business(X_holdout_all)
metrics_lgbm = evaluate_on_holdout(y_holdout_all, scores_lgbm, label="PU-LightGBM")

## 4. Сравнение моделей: PR-кривые и ROC

In [ ]:
all_scores = {
    "LogReg Baseline": scores_baseline,
    "PU-LightGBM": scores_lgbm,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_pr_curve(y_holdout_all, all_scores, ax=axes[0])
plot_roc_curve(y_holdout_all, all_scores, ax=axes[1])
plt.tight_layout()
plt.show()

# Score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_score_distribution(
    scores_lgbm[:len(X_pos_holdout)],
    scores_lgbm[len(X_pos_holdout):],
    threshold=DEFAULT_THRESHOLD,
    ax=axes[0],
)
axes[0].set_title("PU-LightGBM Score Distribution")

plot_confusion_matrix(metrics_lgbm["confusion_matrix"], title="PU-LightGBM Confusion Matrix", ax=axes[1])
plt.tight_layout()
plt.show()

## 5. Precision@K — бизнес-метрика

In [ ]:
pk_lgbm = precision_at_k_table(y_holdout_all, scores_lgbm)
pk_baseline = precision_at_k_table(y_holdout_all, scores_baseline)

print("PU-LightGBM:")
print(pk_lgbm.to_string(index=False))
print("\nLogReg Baseline:")
print(pk_baseline.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(pk_lgbm["K"], pk_lgbm["Precision@K"], marker="o", label="PU-LightGBM")
ax.plot(pk_baseline["K"], pk_baseline["Precision@K"], marker="s", label="LogReg Baseline")
ax.set_xlabel("K (top-K cards)")
ax.set_ylabel("Precision@K")
ax.set_title("Precision@K — Business cards in top-K")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 6. Synthetic Injection Test — оценка recall

Берём 1000 business holdout-карт, вколоть в consumer pool, смотрим сколько попало в top-5%.

In [ ]:
inject_result = run_injection_test(
    pu_model=pu_lgbm,
    X_pos_holdout=X_pos_holdout,
    X_unlabeled=X_unlabeled,
    n_inject=1000,
    top_pct=0.05,
    verbose=True,
)

# Visualize injected vs non-injected score distribution
inj_scores = inject_result["scores"]
inj_flags = inject_result["inject_flags"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(inj_scores[inj_flags], bins=50, alpha=0.7, label="Injected business cards", color="#636EFA")
ax.hist(inj_scores[~inj_flags], bins=50, alpha=0.5, label="Consumer pool", color="#EF553B")
top_k_threshold = np.sort(inj_scores)[::-1][inject_result["top_k"]]
ax.axvline(top_k_threshold, color="black", linestyle="--",
           label=f"Top-5% threshold ({top_k_threshold:.3f})")
ax.set_xlabel("Business Score")
ax.set_ylabel("Count (log scale)")
ax.set_yscale("log")
ax.set_title(f"Injection Test — Recall = {inject_result['recall']:.3f}")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 7. SHAP — объяснение модели

In [ ]:
# SHAP на последней (или средней) модели из PU-bagging
# Используем sample для скорости
sample_size = min(2000, len(X_holdout_all))
X_shap = X_holdout_all.sample(sample_size, random_state=42)

explainer = shap.TreeExplainer(pu_lgbm.models_[-1])
shap_values = explainer.shap_values(X_shap)

# Если список (бинарная классификация) — берём класс 1
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

plt.figure()
shap.summary_plot(sv, X_shap, plot_type="bar", show=False, max_display=20)
plt.title("SHAP Feature Importance (PU-LightGBM)")
plt.tight_layout()
plt.show()

plt.figure()
shap.summary_plot(sv, X_shap, show=False, max_display=20)
plt.title("SHAP Beeswarm (PU-LightGBM)")
plt.tight_layout()
plt.show()

## 8. Скоринг всех consumer-карт → hidden entrepreneurs

In [ ]:
import polars as pl
from src.config import PROCESSED_DIR

# OOB-скоры уже есть для consumer из fit()
consumer_scores = pu_lgbm.scores_

# Собираем consumer карты с их скорами
consumer_fm = fm.filter(pl.col("label") == 0).to_pandas()
consumer_fm = consumer_fm.reset_index(drop=True)
consumer_fm["business_score"] = consumer_scores

# Hidden entrepreneurs по порогу
threshold = DEFAULT_THRESHOLD
hidden = consumer_fm[consumer_fm["business_score"] >= threshold]
print(f"Threshold: {threshold}")
print(f"Hidden entrepreneurs identified: {len(hidden):,} / {len(consumer_fm):,} = {len(hidden)/len(consumer_fm)*100:.1f}%")

# Сохраняем для сегментации
scored_path = PROCESSED_DIR / "consumer_scored.parquet"
pl.from_pandas(consumer_fm[["card_number", "business_score"] + feature_cols]).write_parquet(scored_path)
print(f"Saved → {scored_path}")

# Score distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(consumer_scores, bins=100, color="#EF553B", alpha=0.7)
ax.axvline(threshold, color="black", linestyle="--", label=f"threshold={threshold}")
ax.set_xlabel("Business Score")
ax.set_ylabel("# Consumer Cards")
ax.set_title(f"Consumer Card Score Distribution  |  {len(hidden):,} candidates")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
print("=== ИТОГИ ===")
print(f"Baseline  ROC-AUC={metrics_baseline['roc_auc']:.4f}  PR-AUC={metrics_baseline['pr_auc']:.4f}")
print(f"PU-LightGBM  ROC-AUC={metrics_lgbm['roc_auc']:.4f}  PR-AUC={metrics_lgbm['pr_auc']:.4f}")
print(f"Injection Test Recall@top5%: {inject_result['recall']:.3f}")
print(f"\nHidden entrepreneurs: {len(hidden):,} карт (threshold={threshold})")
print("\nNext: 04_segmentation.ipynb — KMeans сегментация")